# 04 - SQL Database and Business Analysis

## Objective

The objective of this notebook is to create an SQLite
database using the cleaned DataCo dataset and perform
business analysis using SQL queries.

We will investigate delivery performance, shipping modes,
regional performance, product categories and sales.

In [4]:
# importing libraries

import pandas as pd
import sqlite3
import os

In [10]:
#importing the cleaned file 

csv_path = "../data/processed/supply_chain_cleaned.csv"

df = pd.read_csv(
    csv_path,
    parse_dates=[
        "order date (DateOrders)",
        "shipping date (DateOrders)"
    ]
)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

Dataset loaded successfully!
Dataset shape: (180519, 40)


In [11]:
# saving the database 
db_path = "../data/processed/supply_chain.db"

conn = sqlite3.connect(db_path)

print("SQLite database connected successfully!")

SQLite database connected successfully!


In [12]:
# removing unwanted spaces i order for convience in sql
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-zA-Z0-9]+", "_", regex=True)
    .str.strip("_")
)

print(df.columns.tolist())

['type', 'days_for_shipping_real', 'days_for_shipment_scheduled', 'delivery_status', 'late_delivery_risk', 'category_name', 'customer_city', 'customer_country', 'customer_id', 'customer_segment', 'customer_state', 'department_id', 'department_name', 'latitude', 'longitude', 'market', 'order_city', 'order_country', 'order_customer_id', 'order_date_dateorders', 'order_id', 'order_item_discount', 'order_item_discount_rate', 'order_item_id', 'order_item_product_price', 'order_item_profit_ratio', 'order_item_quantity', 'sales', 'order_item_total', 'order_profit_per_order', 'order_region', 'order_state', 'order_status', 'product_card_id', 'product_category_id', 'product_image', 'product_name', 'shipping_date_dateorders', 'shipping_mode', 'shipping_delay_days']


In [14]:
#creating a table in sqlite 

df.to_sql(
    "supply_chain",
    conn,
    if_exists="replace",
    index=False
)

print("Supply chain table created successfully!")

Supply chain table created successfully!


In [15]:
#verifying the table for sql
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

print(tables)

           name
0  supply_chain


In [16]:
#checking the total records 

query = """
SELECT COUNT(*) AS total_records
FROM supply_chain;
"""

result = pd.read_sql_query(query, conn)

print(result)

   total_records
0         180519


In [17]:
#displaying the first 5 records

query = """
SELECT *
FROM supply_chain
LIMIT 5;
"""

display(pd.read_sql_query(query, conn))

,type,days_for_shipping_real,days_for_shipment_scheduled,delivery_status,late_delivery_risk,category_name,customer_city,customer_country,customer_id,customer_segment,...,order_region,order_state,order_status,product_card_id,product_category_id,product_image,product_name,shipping_date_dateorders,shipping_mode,shipping_delay_days
0,DEBIT,3,4,Advance shipping,0,Sporting Goods,Caguas,Puerto Rico,20755,Consumer,...,Southeast Asia,Java Occidental,COMPLETE,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,2018-02-03 22:56:00,Standard Class,-1
1,TRANSFER,5,4,Late delivery,1,Sporting Goods,Caguas,Puerto Rico,19492,Consumer,...,South Asia,Rajastán,PENDING,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,2018-01-18 12:27:00,Standard Class,1
2,CASH,4,4,Shipping on time,0,Sporting Goods,San Jose,EE. UU.,19491,Consumer,...,South Asia,Rajastán,CLOSED,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,2018-01-17 12:06:00,Standard Class,0
3,DEBIT,3,4,Advance shipping,0,Sporting Goods,Los Angeles,EE. UU.,19490,Home Office,...,Oceania,Queensland,COMPLETE,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,2018-01-16 11:45:00,Standard Class,-1
4,PAYMENT,2,4,Advance shipping,0,Sporting Goods,Caguas,Puerto Rico,19489,Corporate,...,Oceania,Queensland,PENDING_PAYMENT,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,2018-01-15 11:24:00,Standard Class,-2


In [18]:
#  counting unique orders 
query = """
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT order_id) AS unique_orders,
    COUNT(DISTINCT order_item_id) AS unique_order_items
FROM supply_chain;
"""

result = pd.read_sql_query(query, conn)

display(result)

,total_records,unique_orders,unique_order_items
0,180519,65752,180519


# 4.1  SQL Business Analysis

In this section, we use SQL queries to answer
important business questions related to supply
chain delivery performance and sales.

In [19]:
#Overall business performance

query = """
SELECT
    COUNT(*) AS total_order_items,

    COUNT(DISTINCT order_id) AS unique_orders,

    COUNT(DISTINCT customer_id) AS unique_customers,

    ROUND(SUM(sales), 2) AS total_sales,

    ROUND(SUM(order_item_total), 2) AS total_order_item_value,

    ROUND(AVG(order_item_total), 2) AS average_item_value,

    SUM(
        CASE
            WHEN delivery_status = 'Late delivery'
            THEN 1
            ELSE 0
        END
    ) AS late_delivery_records

FROM supply_chain;
"""

result = pd.read_sql_query(query, conn)

display(result)

,total_order_items,unique_orders,unique_customers,total_sales,total_order_item_value,average_item_value,late_delivery_records
0,180519,65752,20652,36784735.01,33054402.38,183.11,98977


In [20]:
# delivery status distribution 

query = """
SELECT
    delivery_status,

    COUNT(*) AS total_records,

    ROUND(
        COUNT(*) * 100.0 /
        (SELECT COUNT(*) FROM supply_chain),
        2
    ) AS percentage

FROM supply_chain

GROUP BY delivery_status

ORDER BY total_records DESC;
"""

result = pd.read_sql_query(query, conn)

display(result)

,delivery_status,total_records,percentage
0,Late delivery,98977,54.83
1,Advance shipping,41592,23.04
2,Shipping on time,32196,17.84
3,Shipping canceled,7754,4.30


In [21]:
# late delivery rate 

query = """
SELECT

    COUNT(*) AS total_records,

    SUM(
        CASE
            WHEN delivery_status = 'Late delivery'
            THEN 1
            ELSE 0
        END
    ) AS late_records,

    ROUND(
        SUM(
            CASE
                WHEN delivery_status = 'Late delivery'
                THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS late_delivery_percentage

FROM supply_chain

WHERE delivery_status != 'Shipping canceled';
"""

result = pd.read_sql_query(query, conn)

display(result)

,total_records,late_records,late_delivery_percentage
0,172765,98977,57.29


In [22]:
# shipping mode performance

query = """
SELECT

    shipping_mode,

    COUNT(*) AS total_records,

    SUM(
        CASE
            WHEN delivery_status = 'Late delivery'
            THEN 1
            ELSE 0
        END
    ) AS late_records,

    ROUND(
        SUM(
            CASE
                WHEN delivery_status = 'Late delivery'
                THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS late_delivery_rate

FROM supply_chain

WHERE delivery_status != 'Shipping canceled'

GROUP BY shipping_mode

ORDER BY late_delivery_rate DESC;
"""

result = pd.read_sql_query(query, conn)

display(result)

,shipping_mode,total_records,late_records,late_delivery_rate
0,First Class,26513,26513,100.00
1,Second Class,33806,26987,79.83
2,Same Day,9293,4454,47.93
3,Standard Class,103153,41023,39.77


In [23]:
# regional delivery performance 

query = """
SELECT

    order_region,

    COUNT(*) AS total_records,

    SUM(
        CASE
            WHEN delivery_status = 'Late delivery'
            THEN 1
            ELSE 0
        END
    ) AS late_records,

    ROUND(
        SUM(
            CASE
                WHEN delivery_status = 'Late delivery'
                THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS late_delivery_rate

FROM supply_chain

WHERE delivery_status != 'Shipping canceled'

GROUP BY order_region

ORDER BY late_delivery_rate DESC;
"""

result = pd.read_sql_query(query, conn)

display(result)

,order_region,total_records,late_records,late_delivery_rate
0,Central Africa,1616,972,60.15
1,Western Europe,25867,15140,58.53
2,South of USA,3855,2256,58.52
3,South Asia,7455,4350,58.35
4,East of USA,6617,3849,58.17
5,East Africa,1781,1036,58.17
6,Southeast Asia,9136,5297,57.98
7,West Asia,5746,3322,57.81
8,Eastern Europe,3785,2182,57.65
9,US Center,5653,3252,57.53


In [24]:
# shpping mode vs region

query = """
SELECT

    order_region,

    shipping_mode,

    COUNT(*) AS total_records,

    SUM(
        CASE
            WHEN delivery_status = 'Late delivery'
            THEN 1
            ELSE 0
        END
    ) AS late_records,

    ROUND(
        SUM(
            CASE
                WHEN delivery_status = 'Late delivery'
                THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS late_delivery_rate

FROM supply_chain

WHERE delivery_status != 'Shipping canceled'

GROUP BY
    order_region,
    shipping_mode

ORDER BY
    order_region,
    late_delivery_rate DESC;
"""

result = pd.read_sql_query(query, conn)

display(result)

,order_region,shipping_mode,total_records,late_records,late_delivery_rate
0,Canada,First Class,170,170,100.00
1,Canada,Second Class,154,125,81.17
2,Canada,Standard Class,554,166,29.96
3,Canada,Same Day,29,7,24.14
4,Caribbean,First Class,1132,1132,100.00
...,...,...,...,...,...
87,West of USA,Standard Class,4548,1782,39.18
88,Western Europe,First Class,4107,4107,100.00
89,Western Europe,Second Class,5206,4236,81.37
90,Western Europe,Same Day,1411,708,50.18


In [26]:
# product category performance 

query = """
SELECT

    category_name,

    COUNT(*) AS total_order_items,

    SUM(order_item_quantity) AS total_quantity_sold,

    ROUND(SUM(sales), 2) AS total_sales,

    ROUND(SUM(order_item_total), 2) AS total_order_item_value

FROM supply_chain

GROUP BY category_name

ORDER BY total_sales DESC

LIMIT 10;
"""

result = pd.read_sql_query(query, conn)

display(result)

,category_name,total_order_items,total_quantity_sold,total_sales,total_order_item_value
0,Fishing,17325,17325,6929653.69,6226935.29
1,Cleats,24551,73734,4431942.78,3982856.50
2,Camping & Hiking,13729,13729,4118425.57,3700783.63
3,Cardio Equipment,12487,37587,3694843.20,3320250.70
4,Women's Apparel,21035,62956,3147800.00,2828708.50
5,Water Sports,15540,15540,3113844.68,2798044.18
6,Men's Footwear,22246,22246,2891757.66,2598494.42
7,Indoor/Outdoor Games,19298,57803,2888993.91,2596454.02
8,Shop By Sport,10984,32726,1309522.04,1177185.63
9,Computers,442,442,663000.00,595395.00


In [27]:
# monthly sales trend 

query = """
SELECT

    strftime('%Y-%m', order_date_dateorders) AS order_month,

    COUNT(*) AS total_order_items,

    ROUND(SUM(sales), 2) AS monthly_sales,

    ROUND(AVG(order_item_total), 2) AS average_item_value

FROM supply_chain

GROUP BY order_month

ORDER BY order_month;
"""

result = pd.read_sql_query(query, conn)

display(result)

,order_month,total_order_items,monthly_sales,average_item_value
0,2015-01,5322,1051590.08,177.64
1,2015-02,4729,927009.90,176.15
2,2015-03,5362,1051253.69,176.06
3,2015-04,5126,1014463.28,177.92
4,2015-05,5357,1050478.44,176.21
5,2015-06,5134,1024006.17,179.24
6,2015-07,5299,1038081.19,176.06
7,2015-08,5273,1029494.69,175.46
8,2015-09,5140,1018338.60,177.92
9,2015-10,5302,1049154.27,177.88


In [28]:
# Financial exposure associated with late deliveries

query = """
SELECT

    COUNT(*) AS late_order_items,

    ROUND(SUM(order_item_total), 2)
        AS late_delivery_order_item_value,

    ROUND(AVG(order_item_total), 2)
        AS average_late_item_value,

    ROUND(SUM(order_profit_per_order), 2)
        AS recorded_profit_for_late_items

FROM supply_chain

WHERE delivery_status = 'Late delivery';
"""

result = pd.read_sql_query(query, conn)

display(result)


,late_order_items,late_delivery_order_item_value,average_late_item_value,recorded_profit_for_late_items
0,98977,18082555.3,182.69,2140051.68


In [29]:
# order level analysis 

query = """
SELECT

    order_id,

    COUNT(*) AS number_of_items,

    ROUND(SUM(order_item_total), 2) AS order_value,

    MAX(
        CASE
            WHEN delivery_status = 'Late delivery'
            THEN 1
            ELSE 0
        END
    ) AS contains_late_item

FROM supply_chain

GROUP BY order_id

ORDER BY number_of_items DESC

LIMIT 10;
"""

result = pd.read_sql_query(query, conn)

display(result)

,order_id,number_of_items,order_value,contains_late_item
0,68880,5,931.77,1
1,68873,5,785.31,0
2,68863,5,802.14,0
3,68862,5,1460.02,1
4,68858,5,2668.11,0
5,68843,5,1475.39,0
6,68842,5,692.58,0
7,68835,5,753.32,1
8,68827,5,647.82,1
9,68825,5,818.93,1


# Phase :- SQL Analysis

This notebook connects to the SQLite database created during the
data cleaning phase and executes 15 SQL queries stored in
analysis_queries.sql.

The objective is to analyse delivery performance, shipping modes,
regions, product categories, monthly trends, and financial exposure.

In [41]:
#importing the libraries

import sqlite3
import pandas as pd
from pathlib import Path

In [42]:
# connecting to database
DB_PATH = "../data/processed/supply_chain.db"

conn = sqlite3.connect(DB_PATH)

print("Database connected successfully!")

Database connected successfully!


In [43]:
# verifying the database before running

tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

display(tables)

,name
0,supply_chain


In [44]:
# checking the total records

record_count = pd.read_sql_query(
    "SELECT COUNT(*) AS total_records FROM supply_chain;",
    conn
)

display(record_count)

,total_records
0,180519


In [45]:
#loading our sql queries file 

SQL_PATH = "../sql/analysis_queries.sql"

sql_text = Path(SQL_PATH).read_text(encoding="utf-8")

queries = [
    query.strip()
    for query in sql_text.split(";")
    if query.strip()
]

print("Total SQL queries found:", len(queries))

Total SQL queries found: 15


In [46]:
# executing all sql queries file 

sql_results = {}

for i, query in enumerate(queries, start=1):

    result = pd.read_sql_query(query, conn)

    sql_results[f"Query_{i}"] = result

    print(f"\n========== QUERY {i} ==========")

    display(result)


========== QUERY 1 ==========


,total_order_items,unique_orders,unique_order_items,unique_customers,total_sales,total_order_item_value
0,180519,65752,180519,20652,36784735.01,33054402.38



========== QUERY 2 ==========


,delivery_status,total_records,percentage
0,Late delivery,98977,54.83
1,Advance shipping,41592,23.04
2,Shipping on time,32196,17.84
3,Shipping canceled,7754,4.30



========== QUERY 3 ==========


,non_cancelled_records,late_records,late_delivery_percentage
0,172765,98977,57.29


... (20 outputs left out)


========== QUERY 14 ==========


,market,non_cancelled_records,late_records,late_delivery_percentage
0,Europe,48090,27743,57.69
1,USCA,24627,14138,57.41
2,Pacific Asia,39585,22712,57.38
3,LATAM,49309,28044,56.87
4,Africa,11154,6340,56.84



========== QUERY 15 ==========


,order_month,non_cancelled_records,late_records,late_delivery_percentage
0,2015-01,5114,2880,56.32
1,2015-02,4487,2594,57.81
2,2015-03,5127,2936,57.27
3,2015-04,4915,2760,56.15
4,2015-05,5087,2951,58.01
5,2015-06,4908,2778,56.60
6,2015-07,5124,2939,57.36
7,2015-08,5025,2936,58.43
8,2015-09,4926,2914,59.16
9,2015-10,5073,2906,57.28


In [48]:
#exporting all sql queries result properly

output_folder = Path("../reports/phase4_sql_results")

output_folder.mkdir(parents=True, exist_ok=True)

for query_name, result in sql_results.items():

    file_path = output_folder / f"{query_name.lower()}_results.csv"

    result.to_csv(file_path, index=False)

    print("Saved:", file_path)

Saved: ..\reports\phase4_sql_results\query_1_results.csv
Saved: ..\reports\phase4_sql_results\query_2_results.csv
Saved: ..\reports\phase4_sql_results\query_3_results.csv
Saved: ..\reports\phase4_sql_results\query_4_results.csv
Saved: ..\reports\phase4_sql_results\query_5_results.csv
Saved: ..\reports\phase4_sql_results\query_6_results.csv
Saved: ..\reports\phase4_sql_results\query_7_results.csv
Saved: ..\reports\phase4_sql_results\query_8_results.csv
Saved: ..\reports\phase4_sql_results\query_9_results.csv
Saved: ..\reports\phase4_sql_results\query_10_results.csv
Saved: ..\reports\phase4_sql_results\query_11_results.csv
Saved: ..\reports\phase4_sql_results\query_12_results.csv
Saved: ..\reports\phase4_sql_results\query_13_results.csv
Saved: ..\reports\phase4_sql_results\query_14_results.csv
Saved: ..\reports\phase4_sql_results\query_15_results.csv


In [49]:
# closing the database connection

conn.close()

print("Database connection closed successfully!")

Database connection closed successfully!
